In [1]:
!pip install -q scikit-fuzzy

In [2]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

# New Antecedent/Consequent objects hold universe variables and membership functions
road_conditions = ctrl.Antecedent(np.arange(0, 11, 1), 'road_conditions')
traffic_density = ctrl.Antecedent(np.arange(0, 11, 1), 'traffic_density')
gap_in_traffic = ctrl.Antecedent(np.arange(0, 11, 1), 'gap_in_traffic')
speed_limit = ctrl.Consequent(np.arange(0, 101, 1), 'speed_limit')

# Auto-membership function population is possible with .automf(3, 5, or 7)
road_conditions.automf(3)
traffic_density.automf(3)
gap_in_traffic.automf(3)

# Custom membership functions can be built interactively with a familiar, Pythonic API
speed_limit['low'] = fuzz.trimf(speed_limit.universe, [0, 0, 50])
speed_limit['medium'] = fuzz.trimf(speed_limit.universe, [0, 50, 100])
speed_limit['high'] = fuzz.trimf(speed_limit.universe, [50, 100, 100])

ModuleNotFoundError: No module named 'skfuzzy'

In [ ]:
# Fuzzy rules
rule1 = ctrl.Rule(road_conditions['poor'] | traffic_density['good'] | gap_in_traffic['poor'], speed_limit['low'])
rule2 = ctrl.Rule(road_conditions['poor'] & traffic_density['average'] & gap_in_traffic['average'], speed_limit['low'])
rule3 = ctrl.Rule(road_conditions['average'] & traffic_density['good'] & gap_in_traffic['poor'], speed_limit['low'])
rule4 = ctrl.Rule(road_conditions['average'] & traffic_density['average'] & gap_in_traffic['average'], speed_limit['medium'])
rule5 = ctrl.Rule(road_conditions['good'] & traffic_density['good'] & gap_in_traffic['poor'], speed_limit['medium'])
rule6 = ctrl.Rule(road_conditions['good'] & traffic_density['average'] & gap_in_traffic['average'], speed_limit['high'])
rule7 = ctrl.Rule(road_conditions['good'] & traffic_density['poor'] & gap_in_traffic['good'], speed_limit['high'])

# Control System Creation and Simulation
speed_ctrl = ctrl.ControlSystem([rule1, rule2, rule3, rule4, rule5, rule6, rule7])
speeding = ctrl.ControlSystemSimulation(speed_ctrl)

In [ ]:
def get_speed_limit(road_conditions_input, traffic_density_input, gap_in_traffic_input):
    # Check if inputs are valid
    if not (0 <= road_conditions_input <= 10 and 0 <= traffic_density_input <= 10 and 0 <= gap_in_traffic_input <= 10):
        return 'Invalid input. All inputs must be between 0 and 10.'

    # Pass inputs to the ControlSystem using Antecedent labels with Pythonic API
    speeding.input['road_conditions'] = road_conditions_input
    speeding.input['traffic_density'] = traffic_density_input
    speeding.input['gap_in_traffic'] = gap_in_traffic_input

    # Crunch the numbers
    speeding.compute()

    return speeding.output['speed_limit']

while True:
    try:
        road_conditions_input = float(input('Enter road conditions (0-10): '))
        traffic_density_input = float(input('Enter traffic density (0-10): '))
        gap_in_traffic_input = float(input('Enter gap in traffic (0-10): '))

        speed_limit_output = get_speed_limit(road_conditions_input, traffic_density_input, gap_in_traffic_input)
        print('Recommended speed limit:', speed_limit_output)

        continue_program = input('Do you want to continue? (yes/no): ')
        if continue_program.lower() != 'yes':
            break
    except Exception as e:
        print('An error occurred:', str(e))